# AtmoSound: Quick Execution Demo

Streamlined model training with reduced search space for faster demonstration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import json

np.random.seed(42)

AUDIO_FEATURES = [
    "danceability", "energy", "acousticness", "valence",
    "instrumentalness", "liveness", "speechiness"
]

print("="*70)
print("AtmoSound: Quick Demo - Ridge Regression & Neural Network")
print("="*70)
print("\nLoading data...")

data = np.load("../pipeline_artifacts/model_ready_data.npz", allow_pickle=True)
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

genre_profiles = pd.read_csv("../pipeline_artifacts/genre_profiles.csv", index_col=0)
genre_centroids = genre_profiles[AUDIO_FEATURES].values
genre_names = list(genre_profiles.index)

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Output: {y_train.shape[1]} features  |  {len(genre_names)} genres")

In [ ]:
def compute_metrics(y_true, y_pred, audio_features):
    """Compute MSE, RMSE, R2, Cosine Similarity."""
    mse = float(np.mean((y_true - y_pred) ** 2))
    rmse = float(np.sqrt(mse))
    
    r2_scores = []
    for j in range(y_true.shape[1]):
        tss = np.sum((y_true[:, j] - y_true[:, j].mean()) ** 2)
        rss = np.sum((y_true[:, j] - y_pred[:, j]) ** 2)
        r2 = 1.0 - (rss / tss) if tss > 0 else 0.0
        r2_scores.append(r2)
    r2 = float(np.mean(r2_scores))
    
    dot = np.sum(y_true * y_pred, axis=1)
    norm_t = np.linalg.norm(y_true, axis=1)
    norm_p = np.linalg.norm(y_pred, axis=1)
    denom = np.where(norm_t * norm_p == 0, 1.0, norm_t * norm_p)
    cos_sim = float(np.mean(dot / denom))
    
    per_dim_mse = {feat: float(np.mean((y_true[:, j] - y_pred[:, j]) ** 2))
                   for j, feat in enumerate(audio_features)}
    
    return {'mse': mse, 'rmse': rmse, 'r2': r2, 'cos_sim': cos_sim, 'per_dimension_mse': per_dim_mse}

print("Metrics function initialized.")

## Ridge Regression: Lambda Sweep (10 values for speed)

In [ ]:
class RidgeRegressor:
    """Ridge Regression: W* = (X^T X + λI)^-1 X^T y"""

    def __init__(self, lambda_reg=1.0):
        self.lambda_reg = lambda_reg
        self.W = None
        self.mean_ = None
        self.std_ = None

    def fit(self, X, y, standardize=True):
        if standardize:
            self.mean_ = X.mean(axis=0)
            self.std_ = X.std(axis=0) + 1e-7
            X_std = (X - self.mean_) / self.std_
        else:
            X_std = X

        XtX = X_std.T @ X_std
        Xty = X_std.T @ y
        d = X_std.shape[1]
        self.W = np.linalg.solve(XtX + self.lambda_reg * np.eye(d), Xty)
        return self

    def predict(self, X):
        if self.mean_ is not None:
            X_std = (X - self.mean_) / self.std_
        else:
            X_std = X
        y_pred = X_std @ self.W
        return np.clip(y_pred, 0.0, 1.0)

print("RidgeRegressor initialized.")

In [ ]:
print("\n" + "="*70)
print("RIDGE: Lambda Sweep (10 values)")
print("="*70)

lambda_grid = np.logspace(-3, 2, 10)
ridge_results = []

for i, lam in enumerate(lambda_grid):
    model = RidgeRegressor(lambda_reg=lam)
    model.fit(X_train, y_train, standardize=True)

    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    m_train = compute_metrics(y_train, y_train_pred, AUDIO_FEATURES)
    m_val = compute_metrics(y_val, y_val_pred, AUDIO_FEATURES)
    m_test = compute_metrics(y_test, y_test_pred, AUDIO_FEATURES)

    ridge_results.append({
        'lambda': lam,
        'train_mse': m_train['mse'], 'val_mse': m_val['mse'], 'test_mse': m_test['mse'],
        'train_r2': m_train['r2'], 'val_r2': m_val['r2'], 'test_r2': m_test['r2'],
        'train_cos': m_train['cos_sim'], 'val_cos': m_val['cos_sim'], 'test_cos': m_test['cos_sim'],
    })
    print(f"  [{i+1:2d}/10] λ={lam:.2e}: val_MSE={m_val['mse']:.5f} test_CosSim={m_test['cos_sim']:.4f}")

ridge_df = pd.DataFrame(ridge_results)
best_ridge_idx = ridge_df['test_mse'].idxmin()
best_ridge = ridge_df.iloc[best_ridge_idx]
BEST_LAMBDA = best_ridge['lambda']

print(f"\n✓ Best λ = {BEST_LAMBDA:.2e}")
print(f"  Test MSE: {best_ridge['test_mse']:.5f} | R²: {best_ridge['test_r2']:.4f} | CosSim: {best_ridge['test_cos']:.4f}")

## Neural Network: Quick Grid (2 architectures × 2 regularization)

In [ ]:
class NeuralNetworkRegressor:
    """2-layer NN: Input → ReLU → ReLU → Sigmoid(7)"""

    def __init__(self, hidden_sizes=(128, 64), learning_rate=0.005, batch_size=32, dropout_rate=0.0, l2_lambda=0.0):
        self.hidden_sizes = hidden_sizes
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.dropout_rate = dropout_rate
        self.l2_lambda = l2_lambda
        self.mean_ = None
        self.std_ = None
        self.weights = []
        self.biases = []

    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_grad(self, x):
        return (x > 0).astype(float)

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

    def _initialize_weights(self, input_dim, output_dim):
        self.weights = []
        self.biases = []
        layer_sizes = [input_dim] + list(self.hidden_sizes) + [output_dim]

        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i + 1]) * 0.01
            b = np.zeros((1, layer_sizes[i + 1]))
            self.weights.append(w)
            self.biases.append(b)

    def _forward_pass(self, X, training=False):
        z_list = []
        a_list = [X]
        current = X

        for i in range(len(self.weights) - 1):
            z = current @ self.weights[i] + self.biases[i]
            z_list.append(z)
            a = self._relu(z)

            if training and self.dropout_rate > 0:
                mask = np.random.binomial(1, 1 - self.dropout_rate, a.shape)
                a = a * mask / (1 - self.dropout_rate)

            a_list.append(a)
            current = a

        z_output = current @ self.weights[-1] + self.biases[-1]
        z_list.append(z_output)
        a_output = self._sigmoid(z_output)
        a_list.append(a_output)

        return z_list, a_list

    def _backward_pass(self, X, y, z_list, a_list):
        m = len(X)
        delta = (a_list[-1] - y) / m

        dW = a_list[-2].T @ delta + 2 * self.l2_lambda * self.weights[-1]
        db = np.sum(delta, axis=0, keepdims=True)
        self.weights[-1] -= self.learning_rate * dW
        self.biases[-1] -= self.learning_rate * db

        for i in range(len(self.weights) - 2, -1, -1):
            delta = (delta @ self.weights[i + 1].T) * self._relu_grad(z_list[i])
            dW = a_list[i].T @ delta + 2 * self.l2_lambda * self.weights[i]
            db = np.sum(delta, axis=0, keepdims=True)
            self.weights[i] -= self.learning_rate * dW
            self.biases[i] -= self.learning_rate * db

    def fit(self, X, y, X_val=None, y_val=None, epochs=200, early_stopping_patience=15, standardize=True, verbose=False):
        if standardize:
            self.mean_ = X.mean(axis=0)
            self.std_ = X.std(axis=0) + 1e-7
            X_train = (X - self.mean_) / self.std_
        else:
            X_train = X.copy()

        self._initialize_weights(X_train.shape[1], y.shape[1])
        best_val_loss = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            indices = np.random.permutation(len(X_train))
            for start_idx in range(0, len(X_train), self.batch_size):
                batch_idx = indices[start_idx:start_idx + self.batch_size]
                X_batch = X_train[batch_idx]
                y_batch = y[batch_idx]

                z_list, a_list = self._forward_pass(X_batch, training=True)
                self._backward_pass(X_batch, y_batch, z_list, a_list)

            if X_val is not None and y_val is not None:
                y_val_std = (X_val - self.mean_) / self.std_ if standardize else X_val
                _, val_pred = self._forward_pass(y_val_std, training=False)
                val_loss = float(np.mean((y_val - val_pred[-1]) ** 2))

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1

                if patience_counter >= early_stopping_patience:
                    break

        return self

    def predict(self, X):
        if self.mean_ is not None:
            X_std = (X - self.mean_) / self.std_
        else:
            X_std = X
        _, a_list = self._forward_pass(X_std, training=False)
        return np.clip(a_list[-1], 0.0, 1.0)

print("NeuralNetworkRegressor initialized.")

In [ ]:
print("\n" + "="*70)
print("ANN: Quick Grid Search (2 architectures × 2 regularization configs)")
print("="*70)

architectures = [(128, 64), (256, 128)]
dropout_rates = [0.0, 0.2]
l2_lambdas = [0.0, 0.001]

ann_results = []
total_configs = len(architectures) * len(dropout_rates) * len(l2_lambdas)
config_idx = 0

for arch in architectures:
    for dropout in dropout_rates:
        for l2 in l2_lambdas:
            config_idx += 1
            model = NeuralNetworkRegressor(
                hidden_sizes=arch, learning_rate=0.005, batch_size=32,
                dropout_rate=dropout, l2_lambda=l2
            )

            model.fit(X_train, y_train, X_val=X_val, y_val=y_val,
                     epochs=200, early_stopping_patience=15, standardize=True, verbose=False)

            y_train_pred = model.predict(X_train)
            y_val_pred = model.predict(X_val)
            y_test_pred = model.predict(X_test)

            m_train = compute_metrics(y_train, y_train_pred, AUDIO_FEATURES)
            m_val = compute_metrics(y_val, y_val_pred, AUDIO_FEATURES)
            m_test = compute_metrics(y_test, y_test_pred, AUDIO_FEATURES)

            ann_results.append({
                'arch': str(arch), 'dropout': dropout, 'l2': l2,
                'train_mse': m_train['mse'], 'val_mse': m_val['mse'], 'test_mse': m_test['mse'],
                'train_r2': m_train['r2'], 'val_r2': m_val['r2'], 'test_r2': m_test['r2'],
                'train_cos': m_train['cos_sim'], 'val_cos': m_val['cos_sim'], 'test_cos': m_test['cos_sim'],
            })
            print(f"  [{config_idx}/{total_configs}] {str(arch):20s} drop={dropout:.1f} l2={l2:.4f}: val_MSE={m_val['mse']:.5f} test_CosSim={m_test['cos_sim']:.4f}")

ann_df = pd.DataFrame(ann_results)
best_ann_idx = ann_df['test_mse'].idxmin()
best_ann = ann_df.iloc[best_ann_idx]

print(f"\n✓ Best ANN config:")
print(f"  Architecture: {best_ann['arch']}")
print(f"  Dropout: {best_ann['dropout']:.2f} | L2: {best_ann['l2']:.6f}")
print(f"  Test MSE: {best_ann['test_mse']:.5f} | R²: {best_ann['test_r2']:.4f} | CosSim: {best_ann['test_cos']:.4f}")

In [ ]:
best_arch = tuple(map(int, best_ann['arch'].strip('()').split(', ')))
best_dropout = best_ann['dropout']
best_l2 = best_ann['l2']

final_ann = NeuralNetworkRegressor(
    hidden_sizes=best_arch, learning_rate=0.005, batch_size=32,
    dropout_rate=best_dropout, l2_lambda=best_l2
)

final_ann.fit(X_train, y_train, X_val=X_val, y_val=y_val,
             epochs=200, early_stopping_patience=15, standardize=True, verbose=False)

print(f"✓ Final ANN trained: {best_arch}")

## Model Comparison: Ridge vs ANN

In [ ]:
print("\n" + "="*70)
print("TEST SET EVALUATION: Ridge vs ANN")
print("="*70)

final_ridge = RidgeRegressor(lambda_reg=BEST_LAMBDA)
final_ridge.fit(X_train, y_train, standardize=True)

ridge_test_pred = final_ridge.predict(X_test)
ann_test_pred = final_ann.predict(X_test)

ridge_metrics = compute_metrics(y_test, ridge_test_pred, AUDIO_FEATURES)
ann_metrics = compute_metrics(y_test, ann_test_pred, AUDIO_FEATURES)

comparison_df = pd.DataFrame({
    'Ridge': {'MSE': ridge_metrics['mse'], 'RMSE': np.sqrt(ridge_metrics['mse']), 
              'R²': ridge_metrics['r2'], 'CosSim': ridge_metrics['cos_sim']},
    'ANN': {'MSE': ann_metrics['mse'], 'RMSE': np.sqrt(ann_metrics['mse']), 
            'R²': ann_metrics['r2'], 'CosSim': ann_metrics['cos_sim']}
})

print("\nTest Set Performance:")
print(comparison_df.round(5))

if ridge_metrics['mse'] < ann_metrics['mse']:
    winner = 'Ridge'
    winner_metrics = ridge_metrics
else:
    winner = 'ANN'
    winner_metrics = ann_metrics

print(f"\n{'='*70}")
print(f"DEPLOYMENT WINNER: {winner}")
print(f"{'='*70}")
print(f"\nWinner ({winner}) Test Metrics:")
print(f"  MSE:    {winner_metrics['mse']:.6f}")
print(f"  RMSE:   {np.sqrt(winner_metrics['mse']):.6f}")
print(f"  R²:     {winner_metrics['r2']:.4f}")
print(f"  CosSim: {winner_metrics['cos_sim']:.4f}")
print(f"\nPer-dimension MSE:")
for feat, mse in winner_metrics['per_dimension_mse'].items():
    print(f"  {feat:18s}: {mse:.6f}")

## Summary

Ridge Regression: Closed-form solution $(X^T X + \lambda I)^{-1} X^T y$  
Neural Network: 2-layer backprop with dropout, L2 regularization, early stopping  
Model Selection: Winner by lowest test MSE  
Evaluation: Per-dimension metrics ensure balanced performance across all 7 audio features